In [ ]:
!pip install darts -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path

from darts import TimeSeries
from sklearn.preprocessing import MinMaxScaler
from darts.models import RNNModel
from darts.metrics import rmse, mae, mape
from pytorch_lightning.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import torch
torch.set_float32_matmul_precision("high")

In [ ]:
DATA_DIR   = Path("/content/drive/MyDrive/Thesis/Datasets/network_anomaly")
INSTITUTIONS_DIR = DATA_DIR / "institutions/agg_1_hour"

TIMES_PATH = DATA_DIR / "times/times_1_hour.csv"
FEATURE = "n_bytes"
FEATURE_COLS = [FEATURE]

times_df = pd.read_csv(TIMES_PATH)

times_df["time"] = pd.to_datetime(times_df["time"], errors="coerce")
if times_df["time"].dt.tz is not None:
    times_df["time"] = times_df["time"].dt.tz_localize(None)

times_df["id_time"] = times_df["id_time"].astype(int)

from darts import TimeSeries

raw_series = {}
ts_dict    = {}

for csv_path in INSTITUTIONS_DIR.glob("*.csv"):
    sid = csv_path.stem
    df = pd.read_csv(csv_path)

    df["id_time"] = df["id_time"].astype(int)

    df = df.merge(times_df, on="id_time", how="left")

    df = df.sort_values("time")
    df = df.set_index("time")

    #keep only n_bytes
    df = df[FEATURE_COLS].copy()

    raw_series[sid] = df

    ts = TimeSeries.from_dataframe(
        df,
        value_cols=FEATURE_COLS,  # this is just ["n_bytes"]
        fill_missing_dates=True,
        freq="h"
    )
    ts_dict[sid] = ts

print(f"Loaded {len(ts_dict)} institutions into ts_dict")



In [ ]:
from darts.dataprocessing.transformers import MissingValuesFiller
import numpy as np

filler = MissingValuesFiller()

filled_ts_dict = {}

for sid, ts in ts_dict.items():
    ts_filled = filler.transform(ts)
    n_nans = np.isnan(ts_filled.values()).sum()
    print(f"{sid}: NaNs AFTER filling = {n_nans}")
    filled_ts_dict[sid] = ts_filled


In [ ]:
train_ts_dict = {}
val_ts_dict   = {}
test_ts_dict  = {}

for sid, ts in filled_ts_dict.items():
    n = len(ts)
    train_end = int(n * 0.35)
    val_end   = int(n * 0.40)  # 35% + 5%

    train_ts_dict[sid] = ts[:train_end]
    val_ts_dict[sid]   = ts[train_end:val_end]
    test_ts_dict[sid]  = ts[val_end:]


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

train_scaled_log = {}
val_scaled_log   = {}
test_scaled_log  = {}
scalers_log      = {}   # per-subnet scaler in log-space

for sid in train_ts_dict.keys():
    train_ts = train_ts_dict[sid]
    val_ts   = val_ts_dict[sid]
    test_ts  = test_ts_dict[sid]

    # raw values (n_bytes)
    train_vals = train_ts.values()   # (T_train, 1)
    val_vals   = val_ts.values()
    test_vals  = test_ts.values()

    # ---- log1p transform ----
    train_log = np.log1p(train_vals)
    val_log   = np.log1p(val_vals)
    test_log  = np.log1p(test_vals)

    # concatenate train+val in log-space
    tv_log = np.concatenate([train_log, val_log], axis=0)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(tv_log)

    train_scaled_vals = scaler.transform(train_log)
    val_scaled_vals   = scaler.transform(val_log)
    test_scaled_vals  = scaler.transform(test_log)

    # wrap back into TimeSeries
    train_scaled_log[sid] = train_ts.with_values(train_scaled_vals)
    val_scaled_log[sid]   = val_ts.with_values(val_scaled_vals)
    test_scaled_log[sid]  = test_ts.with_values(test_scaled_vals)

    scalers_log[sid] = scaler

valid_ids = []

In [ ]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=7,        # stop if no improvement for 5 epochs
    min_delta= 1e-4,
    mode="min",

)
early_stop_lstm = EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=1e-4,
    mode="min",
)

In [ ]:
from darts.models import RNNModel
from darts.metrics import rmse, r2_score
from pytorch_lightning.callbacks import EarlyStopping
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- CONFIG ---
MODEL_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Institutions/GRU_3layer_global")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = MODEL_DIR / "plots_rolling_check"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CHUNK = 168
MIN_LEN = INPUT_CHUNK + 1

# --- 1. FILTER VALID INSTITUTIONS ---
print("Filtering valid Institutions for GRU 3-Layer Training...")
valid_inst_ids = []
train_list = []
val_list = []

#Ensure you are using the Institution dictionaries
for sid, ts in train_scaled_log.items():
    sid_str = str(sid)
    if len(ts) >= MIN_LEN and len(val_scaled_log[sid_str]) >= MIN_LEN:
        valid_inst_ids.append(sid_str)
        train_list.append(train_scaled_log[sid_str])
        val_list.append(val_scaled_log[sid_str])

print(f"✅ Found {len(valid_inst_ids)} valid Institutions.")

# --- 2. TRAIN MODEL (3-Layer) ---
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    min_delta=1e-4,
    mode="min",
)

gru_inst_3l = RNNModel(
    model="GRU",
    input_chunk_length=INPUT_CHUNK,
    training_length=INPUT_CHUNK,
    hidden_dim=64,
    n_rnn_layers=3,          # <--- 3 LAYERS
    dropout=0.1,             # <--- Dropout enabled
    batch_size=64,
    n_epochs=20,
    optimizer_kwargs={"lr": 5e-4},
    random_state=42,
    model_name="gru_institution_168_3layer_log",
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "devices": 1,
        "callbacks": [early_stop],
    },
)

print(f"Starting GRU 3-Layer Training on {len(valid_inst_ids)} Institutions...")
gru_inst_3l.fit(
    series=train_list,
    val_series=val_list,
    verbose=True,
)

# Save the model
save_path = MODEL_DIR / "gru_168_3layer_institution_log.pth.tar"
gru_inst_3l.save(str(save_path))
print(f"GRU 3-Layer Institution Model saved to: {save_path}")

# --- 3. ROLLING EVALUATION ---
print(f"\nStarting ROLLING evaluation...")
results_inst = []

for i, sid in enumerate(valid_inst_ids):

    train_ts = train_scaled_log[sid]
    val_ts   = val_scaled_log[sid]
    test_ts  = test_scaled_log[sid]

    full_series = train_ts.concatenate(val_ts).concatenate(test_ts)

    try:
        pred = gru_inst_3l.historical_forecasts(
            series=full_series,
            start=test_ts.start_time(),
            forecast_horizon=1,
            stride=1,
            retrain=False,
            verbose=False,
            last_points_only=True
        )

        r2_val = r2_score(test_ts, pred)
        rmse_val = rmse(test_ts, pred)

        results_inst.append({
            "institution": sid,
            "R2_LogScaled": r2_val,
            "RMSE_LogScaled": rmse_val,
            "n_test_points": len(test_ts)
        })

        if i < 3:
            plt.figure(figsize=(12, 5))
            plt.plot(test_ts.time_index, test_ts.values(), label="Actual (Log)", color='blue', alpha=0.5)
            plt.plot(pred.time_index, pred.values(), label="GRU 3L Pred", color='purple', alpha=0.8)
            plt.title(f"Institution GRU (3L) | Inst {sid} | R2: {r2_val:.4f}")
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.savefig(PLOT_DIR / f"gru_3layer_inst_{sid}.png")
            plt.close()

    except Exception as e:
        print(f"Error on Inst {sid}: {e}")

# --- 4. SAVE RESULTS ---
if results_inst:
    df_inst = pd.DataFrame(results_inst)
    print("\n=== Institution GRU 3-Layer Results ===")
    print(df_inst[["R2_LogScaled", "RMSE_LogScaled"]].mean())

    csv_path = MODEL_DIR / "gru_3layer_institution_rolling_metrics.csv"
    df_inst.to_csv(csv_path, index=False)
    print(f"\n✅ Saved Institution results to: {csv_path}")

In [ ]:
from darts.models import RNNModel
from darts.metrics import rmse, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

MODEL_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Institutions/LSTM_3layer_global")

# --- STEP 1: LOAD THE SAVED MODEL ---
LOAD_PATH = MODEL_DIR / "lstm_168_3layer_institutions_global.pth.tar"

print(f"Attempting to load model from: {LOAD_PATH}")
lstm_loaded = RNNModel.load(str(LOAD_PATH))
print("Model loaded successfully!")

# Create directory for plots
PLOT_DIR = MODEL_DIR / "plots_rolling_check"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Starting ROLLING evaluation on {len(valid_inst_ids)} institutions...")

results_inst = []

for i, sid in enumerate(valid_inst_ids):

    # 1. Get the Full Series (Train + Val + Test)
    train_ts_s = train_scaled_log[sid]
    val_ts_s   = val_scaled_log[sid]
    test_ts_s  = test_scaled_log[sid]

    full_series_s = train_ts_s.concatenate(val_ts_s).concatenate(test_ts_s)

    # 2. Generate Rolling Forecasts
    pred_s = lstm_loaded.historical_forecasts(
        series=full_series_s,
        start=test_ts_s.start_time(),
        forecast_horizon=1,
        stride=1,
        retrain=False,
        verbose=False,
        last_points_only=True
    )

    # 3. Calculate Metrics on LOG Data
    rmse_log = rmse(test_ts_s, pred_s)
    r2_log   = r2_score(test_ts_s, pred_s)

    results_inst.append({
        "institution": sid,
        "R2_LogScaled": r2_log,
        "RMSE_LogScaled": rmse_log,
        "n_test_points": len(test_ts_s),
    })

    # 4. Plotting (sanity check)
    if r2_log < 0 or i < 3:
        plt.figure(figsize=(12, 5))
        plt.plot(test_ts_s.time_index, test_ts_s.values(), label="Actual (Log)", color='blue', alpha=0.5)
        plt.plot(pred_s.time_index, pred_s.values(), label="Rolling Prediction (Log)", color='orange', alpha=0.8)

        plt.title(f"Rolling Forecast | Inst: {sid} | R2: {r2_log:.4f}")
        plt.legend()
        plt.grid(True, alpha=0.3)

        plot_path = PLOT_DIR / f"rolling_eval_{sid}.png"
        plt.savefig(plot_path)
        plt.close()

        if i < 3:
            print(f"[{i+1}] {sid}: R2={r2_log:.4f} -> Plot saved.")

# --- Summary & SAVE CSV ---
df_results = pd.DataFrame(results_inst)

print("\n--- Rolling Forecast Results (Log Scaled) ---")
print(df_results[["institution", "R2_LogScaled", "RMSE_LogScaled"]].head(10))
print(f"\nAverage R2 (Log Scaled): {df_results['R2_LogScaled'].mean():.4f}")

# Save the CSV so you can compare models later
CSV_PATH = MODEL_DIR / "lstm_3layer_rolling_metrics.csv"
df_results.to_csv(CSV_PATH, index=False)
print(f"Results saved to: {CSV_PATH}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- 1. CONFIG ---
base_dir = Path("/content/drive/MyDrive/Thesis/New_Results/Institutions")

save_path = base_dir / "FINAL_INSTITUTION_COMPARISON_PLOT_PRETTY.png"

# Define file paths
files = {
    "Inst LSTM (1-Layer)": base_dir / "LSTM_1layer_global/lstm_1layer_rolling_metrics.csv",
    "Inst LSTM (3-Layer)": base_dir / "LSTM_3layer_global/lstm_3layer_rolling_metrics.csv",
    "Inst GRU (1-Layer)":  base_dir / "GRU_1layer_global/gru_1layer_institution_rolling_metrics.csv",
    "Inst GRU (3-Layer)":  base_dir / "GRU_3layer_global/gru_3layer_institution_rolling_metrics.csv",
}

# --- 2. LOAD DATA ---
all_results = []
for model_name, file_path in files.items():
    candidates = [
        file_path,
        file_path.parent / "lstm_1layer_institutions_global_rolling_metrics.csv",
        file_path.parent / "lstm_168_1layer_institutions_global.pth.tar_rolling_metrics.csv"
    ]
    for p in candidates:
        if p.exists():
            df = pd.read_csv(p)
            if "R2_LogScaled" in df.columns:
                df = df.rename(columns={"R2_LogScaled": "R2"})
            df["Model"] = model_name
            all_results.append(df)
            break

if not all_results:
    print("No data found! Check your paths.")
else:
    master_df = pd.concat(all_results, ignore_index=True)

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(12, 7))


    ax = sns.boxplot(
        data=master_df,
        x="Model",
        y="R2",
        hue="Model",
        legend=False,
        palette="viridis",
        showfliers=False,
        linewidth=1.5,
        width=0.6
    )

    plt.axhline(0, color='red', linestyle='--', linewidth=1.2, label="Baseline (Mean Predictor)")

    # Titles and Labels
    plt.title("Comparison of Model Accuracy ($R^2$) Across Institutions", fontsize=16, fontweight='bold', pad=20)
    plt.ylabel("$R^2$ Score (Higher is Better)", fontsize=12, labelpad=10)
    plt.xlabel("Model Architecture", fontsize=12, labelpad=10)

    # Tweak ticks and grid
    plt.xticks(rotation=15, fontsize=10)
    plt.yticks(fontsize=10)
    ax.grid(True, axis='y', linestyle='-', alpha=0.5)

    import matplotlib.lines as mlines
    baseline_handle = mlines.Line2D([], [], color='red', linestyle='--', label='Baseline')
    plt.legend(handles=[baseline_handle], loc='upper right', frameon=True)

    plt.tight_layout()

    # Save high-resolution image
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n✅ Pretty boxplot saved to: {save_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- CONFIG ---
base_dir = Path("/content/drive/MyDrive/Thesis/New_Results")
subnet_csv = base_dir / "FINAL_MODEL_COMPARISON_TABLE.csv"
inst_csv   = base_dir / "Institutions/FINAL_INSTITUTION_COMPARISON_TABLE.csv"

# --- LOAD DATA ---
try:
    df_sub = pd.read_csv(subnet_csv, index_col=0, header=[0, 1])
    df_inst = pd.read_csv(inst_csv, index_col=0, header=[0, 1])

    print("✅ Loaded comparison tables with Multi-Index headers.")
except Exception as e:
    print(f"❌ Error loading CSVs: {e}")
    df_sub = None

if df_sub is not None:
    # --- DEFINE CHAMPIONS ---
    champion_sub_name  = "GRU Global (1-Layer Log)"
    champion_inst_name = "Inst GRU (1-Layer)"

    try:

        # 1. Subnet Metrics
        sub_r2 = df_sub.loc[champion_sub_name, ('R2', 'mean')]
        sub_rmse = df_sub.loc[champion_sub_name, ('RMSE', 'mean')]

        # 2. Institution Metrics
        inst_r2 = df_inst.loc[champion_inst_name, ('R2', 'mean')]
        inst_rmse = df_inst.loc[champion_inst_name, ('RMSE', 'mean')]

        # --- BUILD FINAL TABLE ---
        comparison_data = {
            "Data Level": ["Subnets", "Institutions"],
            "Best Model": [champion_sub_name, champion_inst_name],
            "Mean R2": [sub_r2, inst_r2],
            "Mean RMSE": [sub_rmse, inst_rmse]
        }

        final_df = pd.DataFrame(comparison_data)

        print("\n===GRAND COMPARISON TABLE===")
        print(final_df)

        # Save
        final_df.to_csv(base_dir / "GRAND_FINAL_COMPARISON.csv", index=False)

        # --- PLOT BAR CHART ---
        plt.figure(figsize=(8, 6))
        sns.set_theme(style="whitegrid")

        # Assign 'x' to 'hue' to avoid Seaborn warning
        ax = sns.barplot(
            data=final_df,
            x="Data Level",
            y="Mean R2",
            hue="Data Level",
            palette="coolwarm",
            legend=False
        )

        # Formatting
        plt.ylim(0, 0.85)
        plt.title("Impact of Data Aggregation on Predictability", fontsize=14, fontweight='bold')
        plt.ylabel("Mean Accuracy ($R^2$)", fontsize=12)
        plt.xlabel("")

        # Add labels on bars
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', padding=5, fontsize=13, fontweight='bold')

        plt.tight_layout()
        plt.savefig(base_dir / "grand_comparison_plot.png", dpi=300)
        plt.show()

        print("\nGrand Comparison plot saved.")

    except KeyError as e:
        print(f"\nLook-up Error: {e}")
        print("Possible causes:")
        print("1. The Model Name in the script doesn't match the CSV row exactly.")
        print(f"   - Script looked for: '{champion_sub_name}'")
        print(f"   - Subnet CSV contains: {df_sub.index.tolist()}")
        print(f"   - Inst CSV contains: {df_inst.index.tolist()}")